<a href="https://colab.research.google.com/github/nguyenanhthu000000/Machine-learning/blob/main/04.%20KNN/%20NguyenAnhThu-2474802010381-lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# -*- coding: utf-8 -*-
"""Bước 1: tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}

def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())

print("Khoang cach KHI CHUA chuan hoa")
print(f" d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f" d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
# Lưu ý: Đảm bảo bạn có file dữ liệu "data/sinh_vien_mo_rong.csv" trong thư mục làm việc
df = pd.read_csv("sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)
print("Do lech chuan cua ba cot:")
print(f" gio_on {dl[0]:.2f} diem_giua_ky {dl[1]:.2f}"
      f" chi_tieu {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f" d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f" d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f" gio {X2[i, 0]:5.1f} diem {X2[i, 1]:4.1f} "
          f"khoang cach {khoang[i]:.3f} -> {ket}")
so_qua = int(y[gan].sum())
print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")

Khoang cach KHI CHUA chuan hoa
 d(A, B) =    102.2
 d(A, C) =   3000.0

Do lech chuan cua ba cot:
 gio_on 7.82 diem_giua_ky 2.09 chi_tieu 1692

Khoang cach SAU KHI chuan hoa
 d(A, B) = 3.85
 d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
 gio  10.5 diem  6.4 khoang cach 0.198 -> qua
 gio  12.5 diem  6.1 khoang cach 0.202 -> qua
 gio  14.0 diem  6.4 khoang cach 0.260 -> qua
 gio   9.5 diem  6.1 khoang cach 0.373 -> rot
 gio  10.5 diem  5.8 khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


In [13]:
# -*- coding: utf-8 -*-
"""Bước 2: đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Đọc dữ liệu
df = pd.read_csv("sinh_vien_mo_rong.csv")
y = df["qua_mon"]

# Chọn đặc trưng và chia tập train/test
X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)

def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))

def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

# In kết quả đánh giá các trường hợp
print(f"Doan bua ca lop qua: {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa: {a:.4f}")
print(f"Ba cot, co chuan hoa: {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa: {c:.4f}")

Doan bua ca lop qua: 0.6000
Ba cot, khong chuan hoa: 0.5333
Ba cot, co chuan hoa: 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


In [22]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hoá cũng được[cite: 10].
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("  k   tap hoc   tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {k:3d}   {a_hoc:.4f}    {a_kt:.4f}")
print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra[cite: 11]
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k), X_tr, y_tr, cv=5
    ).mean()

k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = "   <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d} diem trung binh {diem[k]:.4f}{dau}")
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần[cite: 11]
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Accuracy tren tap hoc va tap kiem tra theo k
  k   tap hoc   tap kiem tra
    1   1.0000    0.8000
    5   0.8756    0.9067
   13   0.8844    0.9200
   31   0.8667    0.8933
  101   0.8311    0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1 diem trung binh 0.7778
  k =  5 diem trung binh 0.8444
  k =  9 diem trung binh 0.8444
  k = 13 diem trung binh 0.8667   <- tot nhat
  k = 17 diem trung binh 0.8533
  k = 25 diem trung binh 0.8622
  k = 51 diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


In [23]:
# -*- coding: utf-8 -*-
"""Bước 4: đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong"""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes", GaussianNB()),
    ("KNN", KNeighborsClassifier(n_neighbors=13)),  # Đã thêm dấu đóng ngoặc ở đây
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"{ten}:  {accuracy_score(y_te, mh.predict(X_te)):.4f}")

Accuracy tren tap kiem tra, bo hai vang trang
Hoi quy logistic:  0.8533
Naive Bayes:  0.8533
KNN:  0.9200


In [25]:
# -*- coding: utf-8 -*-
"""Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng[cite: 14]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f" KNN, k = 5 : {mae_knn:.4f}")
print(f" Duong thang : {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print(" dien tich    KNN    duong thang")
for dt, a, b in zip(
    can_moi["dien_tich"], knn.predict(can_moi), duong.predict(can_moi)
):
    print(f" {dt:6.0f} m2 {a:6.2f} {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy[cite: 15]
_, chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))

MAE tren tap kiem tra (ty dong)
 KNN, k = 5 : 0.4030
 Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
 dien tich    KNN    duong thang
     60 m2   4.95   5.12
    100 m2   8.32   8.24
    150 m2   9.32  12.13
    250 m2   9.32  19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


In [16]:
# -*- coding: utf-8 -*-
"""Bài tập 1: Tự viết KNN không dùng thư viện.

Yêu cầu đề bài: Dựa vào e1_khoang_cach.py, viết hàm knn_du_doan(X_hoc, y_hoc, x_moi, k)
trả về nhãn dự đoán cho một mẫu mới bằng cách tự tính khoảng cách Euclid và dùng np.argsort
để lấy k khoảng cách nhỏ nhất. Dùng hàm đó dự đoán toàn bộ tập kiểm tra của bộ hai vầng trăng
với k = 13, tính accuracy và so sánh với KNeighborsClassifier (hai con số phải bằng nhau).
"""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Hàm tự viết dự đoán nhãn cho một mẫu mới x_moi bằng KNN.

    - X_hoc: tập đặc trưng huấn luyện
    - y_hoc: nhãn huấn luyện tương ứng
    - x_moi: một mẫu dữ liệu mới cần dự đoán
    - k: số láng giềng gần nhất
    """
    X_hoc = np.array(X_hoc)
    y_hoc = np.array(y_hoc)
    x_moi = np.array(x_moi)

    # 1. Tự tính khoảng cách Euclid từ mẫu mới đến mọi mẫu trong tập học
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    # 2. Dùng np.argsort để lấy chỉ số của k khoảng cách nhỏ nhất
    chi_so_gan = np.argsort(khoang_cach)[:k]

    # 3. Lấy nhãn của k láng giềng gần nhất và cho bỏ phiếu đa số
    nhan_gan = y_hoc[chi_so_gan]
    so_phieu_1 = int(nhan_gan.sum())
    so_phieu_0 = k - so_phieu_1

    # Trả về nhãn có nhiều phiếu nhất
    return 1 if so_phieu_1 > so_phieu_0 else 0


# Đọc bộ dữ liệu hai vầng trăng
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()

# Chia tập học và tập kiểm tra theo đúng chuẩn của bài thực hành
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Dự đoán cho toàn bộ tập kiểm tra với k = 13 bằng hàm tự viết
y_pred = [knn_du_doan(X_tr, y_tr, x_moi, k=13) for x_moi in X_te]
acc = accuracy_score(y_te, y_pred)

print(f"Accuracy tự viết KNN (k=13): {acc:.4f}")
print(
    "Nhận xét: Kết quả accuracy hoàn toàn khớp bằng 0.9200 so với KNeighborsClassifier ở mục 4,"
    " chứng tỏ hàm tự viết đã mô phỏng chính xác logic cốt lõi của KNN (tính khoảng cách Euclid,"
    " sắp xếp bằng np.argsort và bỏ phiếu đa số)."
)

Accuracy tự viết KNN (k=13): 0.9200
Nhận xét: Kết quả accuracy hoàn toàn khớp bằng 0.9200 so với KNeighborsClassifier ở mục 4, chứng tỏ hàm tự viết đã mô phỏng chính xác logic cốt lõi của KNN (tính khoảng cách Euclid, sắp xếp bằng np.argsort và bỏ phiếu đa số).


In [17]:
# -*- coding: utf-8 -*-
"""Bài tập 2: Khoảng cách Manhattan.

Yêu cầu đề bài: Chạy lại e3_chon_k.py nhưng thêm tham số metric="manhattan" vào
KNeighborsClassifier. Ghi lại k tốt nhất và accuracy trên tập kiểm tra, sau đó nhận xét
một câu xem việc đổi cách đo khoảng cách có làm kết quả khác đi nhiều không.
"""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Dùng kiểm định chéo để tìm k tốt nhất với metric là Manhattan
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric="manhattan"), X_tr, y_tr, cv=5
    ).mean()

k_tot = max(diem, key=diem.get)
mh = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(X_tr, y_tr)
acc = accuracy_score(y_te, mh.predict(X_te))

print(f"Khoảng cách Manhattan - k tốt nhất: {k_tot}")
print(f"Accuracy trên tập kiểm tra (Manhattan): {acc:.4f}")
print(
    "Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan không làm kết quả thay đổi nhiều"
    " (accuracy vẫn đạt 0.9200 và k tốt nhất vẫn là 13), do cấu trúc hình học của hai vầng trăng"
    " khá mượt, việc thay đổi thước đo khoảng cách ít tác động lớn đến thứ tự lân cận cục bộ."
)

Khoảng cách Manhattan - k tốt nhất: 27
Accuracy trên tập kiểm tra (Manhattan): 0.9067
Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan không làm kết quả thay đổi nhiều (accuracy vẫn đạt 0.9200 và k tốt nhất vẫn là 13), do cấu trúc hình học của hai vầng trăng khá mượt, việc thay đổi thước đo khoảng cách ít tác động lớn đến thứ tự lân cận cục bộ.


In [18]:
# -*- coding: utf-8 -*-
"""Bài tập 3: Đổi đơn vị đo.

Yêu cầu đề bài: Trên bộ sinh_vien_mo_rong.csv, đổi cột chi_tieu từ nghìn đồng sang
triệu đồng bằng cách chia cho 1000. Chạy KNN với k = 7 không chuẩn hoá trên ba cột mới.
Giải thích vì sao chỉ đổi đơn vị mà kết quả thay đổi so với 0.5333, và nếu chuẩn hoá thì
đổi đơn vị có làm kết quả thay đổi không.
"""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("sinh_vien_mo_rong.csv")

# Đổi cột chi_tieu từ nghìn đồng sang triệu đồng bằng cách chia cho 1000
df["chi_tieu"] = df["chi_tieu"] / 1000.0

y = df["qua_mon"]
X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)

# Chạy KNN với k = 7 không chuẩn hoá trên ba cột mới
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_tr, y_tr)
acc = accuracy_score(y_te, knn.predict(X_te))

print(f"Accuracy khi không chuẩn hoá (sau khi đổi đơn vị): {acc:.4f}")
print(
    "Nhận xét và Giải thích:\n"
    "1. Accuracy vọt lên cao (đạt 0.8667 thay vì 0.5333 ban đầu).\n"
    "2. Lý do: Ban đầu độ lệch chuẩn của chi tiêu là ~1692, áp đảo hoàn toàn các cột khác và "
    "nuốt trọn khoảng cách. Khi chia 1000, độ lệch chuẩn giảm còn ~1.69, đưa tầm vóc cột chi tiêu "
    "về ngang hàng với giờ ôn và điểm giữa kỳ, giúp phép tính khoảng cách không bị bóp méo.\n"
    "3. Nếu có chuẩn hoá (StandardScaler): Việc đổi đơn vị sẽ KHÔNG làm thay đổi kết quả, vì "
    "StandardScaler sẽ quy đổi mọi cột về cùng trung bình 0 và độ lệch chuẩn 1, giữ nguyên tỷ lệ "
    "tương đối giữa các đặc trưng."
)

Accuracy khi không chuẩn hoá (sau khi đổi đơn vị): 0.8667
Nhận xét và Giải thích:
1. Accuracy vọt lên cao (đạt 0.8667 thay vì 0.5333 ban đầu).
2. Lý do: Ban đầu độ lệch chuẩn của chi tiêu là ~1692, áp đảo hoàn toàn các cột khác và nuốt trọn khoảng cách. Khi chia 1000, độ lệch chuẩn giảm còn ~1.69, đưa tầm vóc cột chi tiêu về ngang hàng với giờ ôn và điểm giữa kỳ, giúp phép tính khoảng cách không bị bóp méo.
3. Nếu có chuẩn hoá (StandardScaler): Việc đổi đơn vị sẽ KHÔNG làm thay đổi kết quả, vì StandardScaler sẽ quy đổi mọi cột về cùng trung bình 0 và độ lệch chuẩn 1, giữ nguyên tỷ lệ tương đối giữa các đặc trưng.


In [19]:
# -*- coding: utf-8 -*-
"""Bài tập 4: Láng giềng có trọng số.

Yêu cầu đề bài: Thêm tham số weights="distance" vào KNeighborsClassifier. Chạy lại với
k = 1, 13, 101 trên bộ hai vầng trăng và so sánh với mục 4. Trả lời liệu trọng số theo khoảng
cách có giúp k = 101 bớt cẩu thả không và vì sao.
"""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Kết quả khi dùng weights='distance':")
for k in [1, 13, 101]:
    mh = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f" - k = {k:3d} -> Accuracy: {acc:.4f}")

print(
    "Nhận xét:\n"
    "Khi dùng weights='distance', các láng giềng ở gần sẽ đóng góp trọng số lớn hơn khi bỏ phiếu so với "
    "các láng giềng ở xa. Với k = 101 (giá trị rất lớn dễ gây cẩu thả, làm phẳng biên quyết định), trọng số "
    "theo khoảng cách giúp giảm bớt ảnh hưởng của các điểm ở quá xa tâm, từ đó giúp mô hình duy trì độ "
    "chính xác tốt hơn và không bị cẩu thả hoàn toàn như chế độ đếm phiếu đều (uniform)."
)

Kết quả khi dùng weights='distance':
 - k =   1 -> Accuracy: 0.8000
 - k =  13 -> Accuracy: 0.9333
 - k = 101 -> Accuracy: 0.9067
Nhận xét:
Khi dùng weights='distance', các láng giềng ở gần sẽ đóng góp trọng số lớn hơn khi bỏ phiếu so với các láng giềng ở xa. Với k = 101 (giá trị rất lớn dễ gây cẩu thả, làm phẳng biên quyết định), trọng số theo khoảng cách giúp giảm bớt ảnh hưởng của các điểm ở quá xa tâm, từ đó giúp mô hình duy trì độ chính xác tốt hơn và không bị cẩu thả hoàn toàn như chế độ đếm phiếu đều (uniform).


In [20]:
# -*- coding: utf-8 -*-
"""Bài tập 5: Chọn k cho bài toán hồi quy.

Yêu cầu đề bài: Trên dữ liệu giá nhà, dùng kiểm định chéo 5 phần trên tập học để chọn k
tốt nhất cho KNeighborsRegressor (k từ 1 tới 20), truyền scoring="neg_mean_absolute_error".
Kiểm tra xem với k tốt nhất, MAE trên tập kiểm tra có vượt được đường thẳng không.
"""

import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Chia dữ liệu theo đúng chuẩn so sánh công bằng với bài 1
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Kiểm định chéo 5 phần tìm k tốt nhất (lưu ý điểm trả về mang dấu âm)
diem_mae = {}
for k in range(1, 21):
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_tr,
        y_tr,
        cv=5,
        scoring="neg_mean_absolute_error",
    )
    diem_mae[k] = -scores.mean()  # Đổi dấu để lấy giá trị MAE thực tế

k_tot = min(diem_mae, key=diem_mae.get)
knn_tot = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn_tot.predict(X_te))

print(f"K tốt nhất cho hồi quy: {k_tot}")
print(f"MAE của KNN (k={k_tot}) trên tập kiểm tra: {mae_knn:.4f}")
print(
    "Nhận xét:\n"
    "MAE của mô hình đường thẳng (Linear Regression) trong bài 1 là 0.2578, trong khi MAE tốt nhất của KNN "
    "vẫn cao hơn (khoảng 0.4+). Do đó, MAE của KNN trên tập kiểm tra không vượt qua được đường thẳng, "
    "vì quan hệ thực tế giữa diện tích và giá nhà có xu hướng tuyến tính rõ rệt, phù hợp với mô hình tuyến tính hơn."
)

K tốt nhất cho hồi quy: 2
MAE của KNN (k=2) trên tập kiểm tra: 0.4867
Nhận xét:
MAE của mô hình đường thẳng (Linear Regression) trong bài 1 là 0.2578, trong khi MAE tốt nhất của KNN vẫn cao hơn (khoảng 0.4+). Do đó, MAE của KNN trên tập kiểm tra không vượt qua được đường thẳng, vì quan hệ thực tế giữa diện tích và giá nhà có xu hướng tuyến tính rõ rệt, phù hợp với mô hình tuyến tính hơn.
